In [1]:
# Install TensorFlow into the environment used by this notebook.
# %pip ensures installation happens in the active Jupyter kernel.
%pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


# OR-Gate Neural Network Using TensorFlow

## Step 1 — Import Libraries and Create the Dataset

This notebook uses TensorFlow to create and train a simple neural network that represents an OR logic gate.

NumPy stores the training inputs and expected outputs. TensorFlow creates, trains and tests the neural network.

In [2]:
# Import TensorFlow and give it the standard short name "tf".
# TensorFlow provides the tools required to create and train
# the neural network.
import tensorflow as tf

# Import NumPy and give it the standard short name "np".
# NumPy stores the input features and expected outputs as arrays.
import numpy as np

# Display the installed versions to confirm that both libraries loaded.
print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)

TensorFlow version: 2.21.0
NumPy version: 2.4.6


In [3]:
# Create the four possible input combinations for an OR gate.
#
# Each row is one training example.
# Each example contains two input features: x1 and x2.
#
# float32 is used because TensorFlow normally performs
# neural-network calculations using 32-bit decimal values.
X = np.array([
    [0, 0],  # 0 OR 0
    [0, 1],  # 0 OR 1
    [1, 0],  # 1 OR 0
    [1, 1]   # 1 OR 1
], dtype=np.float32)

# Create the correct output for each input combination.
#
# Each expected output is placed inside its own list because
# the neural network produces one output value per sample.
y = np.array([
    [0],  # 0 OR 0 = 0
    [1],  # 0 OR 1 = 1
    [1],  # 1 OR 0 = 1
    [1]   # 1 OR 1 = 1
], dtype=np.float32)

# Display the completed training dataset.
print("Input data:")
print(X)

print("\nExpected outputs:")
print(y)

# Check the dimensions of both arrays.
print("\nInput shape:", X.shape)
print("Output shape:", y.shape)

Input data:
[[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]

Expected outputs:
[[0.]
 [1.]
 [1.]
 [1.]]

Input shape: (4, 2)
Output shape: (4, 1)


## Step 2 — Create the Neural Network

The network contains one artificial neuron because the OR gate requires one binary output.

The neuron:

1. Receives the two input values, `x1` and `x2`.
2. Multiplies them by learned weights.
3. Adds a learned bias.
4. applies the sigmoid activation function.
5. Returns a value between `0` and `1`.

A result close to `0` represents False, while a result close to `1` represents True.

In [4]:
# Set a random seed so that the model's initial values
# are reproducible when the notebook runs again.
tf.random.set_seed(42)

# Create a sequential neural network.
# Sequential means the data passes through each layer in order.
model = tf.keras.Sequential([
    
    # Define the shape of one input example.
    # Each OR-gate example contains two values: x1 and x2.
    tf.keras.Input(shape=(2,)),

    # Add one densely connected artificial neuron.
    #
    # units=1:
    # The network produces one output value.
    #
    # activation="sigmoid":
    # Converts the neuron's weighted sum into a value
    # between 0 and 1.
    tf.keras.layers.Dense(
        units=1,
        activation="sigmoid"
    )
])

# Display the structure of the neural network.
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3 (12.00 B)

 Trainable params: 3 (12.00 B)

 Non-trainable params: 0 (0.00 B)

## Step 3 — Configure the Model

Before training, TensorFlow needs three settings:

- **Optimizer:** controls how the model updates its weights and bias.
- **Loss function:** measures the difference between the expected and predicted outputs.
- **Metric:** reports how accurately the model classifies the training examples.

This model uses stochastic gradient descent and binary cross-entropy because the OR gate is a binary classification problem.

In [5]:
# Configure the neural network before training.
model.compile(

    # Stochastic Gradient Descent updates the model's
    # weights and bias to reduce prediction errors.
    #
    # learning_rate=0.1 controls the size of each update.
    optimizer=tf.keras.optimizers.SGD(
        learning_rate=0.1
    ),

    # Binary cross-entropy measures the difference between
    # the expected binary answers and predicted probabilities.
    #
    # A smaller loss means the predictions are closer
    # to the expected outputs.
    loss="binary_crossentropy",

    # Ask TensorFlow to calculate classification accuracy
    # while training and evaluating the model.
    metrics=["accuracy"]
)

# Display confirmation because model.compile() does not
# normally produce visible output.
print("The TensorFlow model has been configured successfully.")

The TensorFlow model has been configured successfully.


In [6]:
# Train the neural network using the OR-gate dataset.
training_history = model.fit(

    # Input features containing x1 and x2.
    X,

    # Correct OR-gate outputs.
    y,

    # Process the complete dataset 500 times.
    epochs=500,

    # Hide the progress output for each individual epoch.
    verbose=0
)

# Get the loss recorded during the final epoch.
final_loss = training_history.history["loss"][-1]

# Get the accuracy recorded during the final epoch.
final_accuracy = training_history.history["accuracy"][-1]

# Confirm that training has finished.
print("Training completed successfully.")

# Display the final loss.
# A smaller value means the predictions are closer
# to the expected outputs.
print(f"Final training loss: {final_loss:.4f}")

# Convert the decimal accuracy into a percentage.
print(f"Final training accuracy: {final_accuracy * 100:.1f}%")

Training completed successfully.
Final training loss: 0.1515
Final training accuracy: 100.0%


## Step 5 — Test the Neural Network

The trained model now predicts the output for every OR-gate input.

The sigmoid neuron returns probabilities between `0` and `1`. We convert these probabilities into binary classes using a threshold of `0.5`:

- Probability below `0.5` becomes class `0`.
- Probability of `0.5` or higher becomes class `1`.

In [7]:
# Ask the trained model to predict an output probability
# for every OR-gate input combination.
#
# verbose=0 prevents TensorFlow from displaying a progress bar.
prediction_probabilities = model.predict(X, verbose=0)

# Convert each probability into a binary class.
#
# A probability of 0.5 or higher becomes 1.
# A probability below 0.5 becomes 0.
binary_predictions = (
    prediction_probabilities >= 0.5
).astype(int)

# Print a heading for the results.
print("TensorFlow OR-gate predictions")
print("-" * 48)

# Examine each input and its corresponding results.
for inputs, probability, prediction, expected in zip(
    X,
    prediction_probabilities,
    binary_predictions,
    y
):
    # Separate the two input values.
    x1, x2 = inputs

    # Extract ordinary numbers from the one-element arrays.
    probability_value = probability[0]
    predicted_class = prediction[0]
    expected_class = int(expected[0])

    # Check whether the predicted class is correct.
    is_correct = predicted_class == expected_class
    result = "Correct" if is_correct else "Incorrect"

    # Display one formatted row.
    print(
        f"{int(x1)} OR {int(x2)} | "
        f"Probability: {probability_value:.4f} | "
        f"Prediction: {predicted_class} | "
        f"Expected: {expected_class} | "
        f"{result}"
    )

# Calculate the proportion of correct binary predictions.
test_accuracy = np.mean(binary_predictions == y)

# Display the final accuracy as a percentage.
print(f"\nTest accuracy: {test_accuracy * 100:.1f}%")

TensorFlow OR-gate predictions
------------------------------------------------
0 OR 0 | Probability: 0.3026 | Prediction: 0 | Expected: 0 | Correct
0 OR 1 | Probability: 0.8925 | Prediction: 1 | Expected: 1 | Correct
1 OR 0 | Probability: 0.8835 | Prediction: 1 | Expected: 1 | Correct
1 OR 1 | Probability: 0.9932 | Prediction: 1 | Expected: 1 | Correct

Test accuracy: 100.0%


## Step 6 — Inspect the Learned Parameters

The TensorFlow `Dense` layer learned two weights and one bias during training.

The neuron calculates:

\[
z = (x_1w_1) + (x_2w_2) + b
\]

The sigmoid activation function then converts \(z\) into a probability between `0` and `1`.

Positive weights for both inputs make sense for an OR gate because either active input should increase the probability of class `1`.

In [8]:
# Access the first and only Dense layer in the model.
output_layer = model.layers[0]

# Get the parameters learned by the layer.
#
# learned_weights contains one weight for each input.
# learned_bias contains the neuron's bias value.
learned_weights, learned_bias = output_layer.get_weights()

# Extract the two individual weights.
weight_x1 = learned_weights[0, 0]
weight_x2 = learned_weights[1, 0]

# Extract the single bias value.
bias = learned_bias[0]

# Display the learned parameters.
print("TensorFlow learned parameters")
print("-" * 35)
print(f"Weight for x1: {weight_x1:.4f}")
print(f"Weight for x2: {weight_x2:.4f}")
print(f"Bias:          {bias:.4f}")

TensorFlow learned parameters
-----------------------------------
Weight for x1: 2.8605
Weight for x2: 2.9510
Bias:          -0.8348


## Step 7 — From-Scratch and TensorFlow Comparison

| Learning operation | From-scratch implementation | TensorFlow implementation |
|---|---|---|
| Create the weights | `np.zeros(n_features)` | `Dense(1)` |
| Create the bias | `self.bias = 0.0` | `Dense(1)` |
| Calculate the weighted sum | `np.dot(X, weights) + bias` | Performed by `Dense` |
| Apply the activation function | Manual step function | `activation="sigmoid"` |
| Calculate the error or loss | `target - prediction` | `binary_crossentropy` |
| Update the parameters | Manual perceptron rule | `SGD` optimizer |
| Repeat the training process | Manual epoch loop | `model.fit()` |
| Generate predictions | `predict()` method | `model.predict()` |

The from-scratch implementation exposes every calculation and helps explain how a perceptron learns.

TensorFlow performs the same broad stages through reusable layers, loss functions and optimizers. This makes it easier to construct larger neural networks without manually programming every parameter update.